# Orchestrateur LangGraph du SAV

Classe la demande, délègue à un agent spécialisé, termine dès qu'une réponse finale est disponible.

**Prérequis :** le notebook `rag_server.ipynb` doit tourner (serveur MCP sur `http://localhost:8000/mcp`).

In [ ]:
import json
import os
from datetime import date
from pathlib import Path
from typing import Literal, Optional, TypedDict

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

LLM = ChatOllama(model=os.getenv("OLLAMA_LLM", "qwen2.5:7b"), temperature=0)
MCP_URL = os.getenv("MCP_URL", "http://localhost:8000/mcp")
REGISTRE = Path("donnees") / "garanties.json"
AGENTS = {"documentation": "agent_documentaire", "garantie": "agent_garantie"}

In [ ]:
class Etat(TypedDict):
    question: str
    image_ref: Optional[str]
    categorie: Optional[str]
    agent: Optional[str]
    reponse_finale: Optional[str]

In [ ]:
class Classification(BaseModel):
    categorie: Literal["documentation", "garantie"] = Field(
        description="documentation : dépannage, panne, voyant, pictogramme, utilisation. "
                    "garantie : vérification de la garantie d'un appareil à partir de son numéro de série."
    )

In [ ]:
@tool
def consulter_garantie(numero_serie: str) -> str:
    """Consulte le registre des garanties (lecture seule) pour un numéro de série, ex. PX200-0042."""
    with REGISTRE.open(encoding="utf-8") as f:
        fiche = json.load(f).get(numero_serie.strip().upper())
    if fiche is None:
        return f"Aucun appareil enregistré avec le numéro de série {numero_serie}."
    sous_garantie = date.fromisoformat(fiche["fin_garantie"]) >= date.today()
    return json.dumps({"numero_serie": numero_serie, **fiche, "sous_garantie": sous_garantie,
                       "date_verification": date.today().isoformat()}, ensure_ascii=False)

In [ ]:
async def orchestrateur(etat: Etat) -> dict:
    if etat.get("reponse_finale"):
        return {}
    classification = await LLM.with_structured_output(Classification).ainvoke([
        ("system", "Classe la demande client du SAV dans une seule catégorie."),
        ("user", etat["question"]),
    ])
    return {"categorie": classification.categorie, "agent": AGENTS[classification.categorie]}

In [ ]:
def router(etat: Etat) -> str:
    return END if etat.get("reponse_finale") else etat["agent"]

In [ ]:
async def construire_graphe():
    client = MultiServerMCPClient({"documentation": {"url": MCP_URL, "transport": "streamable_http"}})
    outils_mcp = [t for t in await client.get_tools() if t.name == "rag_multimodal"]

    agent_doc = create_agent(LLM, outils_mcp, system_prompt=(
        "Tu es l'agent documentaire du SAV. Appelle toujours l'outil rag_multimodal avec la question du "
        "client et, si une référence de photo est fournie, avec image_ref. Restitue sa réponse en gardant les sources."
    ))
    agent_gar = create_agent(LLM, [consulter_garantie], system_prompt=(
        "Tu es l'agent garantie du SAV. Extrais le numéro de série de la demande, appelle consulter_garantie, "
        "puis réponds clairement : sous garantie ou non, date de fin de garantie. N'invente aucune donnée."
    ))

    async def agent_documentaire(etat: Etat) -> dict:
        demande = etat["question"]
        if etat.get("image_ref"):
            demande += f"\nRéférence de la photo jointe : {etat['image_ref']}"
        resultat = await agent_doc.ainvoke({"messages": [("user", demande)]})
        return {"reponse_finale": resultat["messages"][-1].content or "Aucune réponse produite."}

    async def agent_garantie(etat: Etat) -> dict:
        resultat = await agent_gar.ainvoke({"messages": [("user", etat["question"])]})
        return {"reponse_finale": resultat["messages"][-1].content or "Aucune réponse produite."}

    graphe = StateGraph(Etat)
    graphe.add_node("orchestrateur", orchestrateur)
    graphe.add_node("agent_documentaire", agent_documentaire)
    graphe.add_node("agent_garantie", agent_garantie)
    graphe.add_edge(START, "orchestrateur")
    graphe.add_conditional_edges("orchestrateur", router, ["agent_documentaire", "agent_garantie", END])
    graphe.add_edge("agent_documentaire", "orchestrateur")
    graphe.add_edge("agent_garantie", "orchestrateur")
    return graphe.compile()

In [ ]:
EXEMPLES = [
    {"question": "Mon imprimante PX-200 affiche un voyant clignotant. Que signifie le pictogramme sur la photo jointe ?",
     "image_ref": "img_px200_01"},
    {"question": "Est-ce que mon imprimante est encore sous garantie ? Son numéro de série est PX200-0042.",
     "image_ref": None},
]

In [ ]:
async def main():
    app = await construire_graphe()
    for exemple in EXEMPLES:
        etat = await app.ainvoke(exemple)
        print(f"\nQuestion  : {etat['question']}")
        print(f"Catégorie : {etat['categorie']} · Agent : {etat['agent']}")
        print(f"Réponse   :\n{etat['reponse_finale']}")

In [ ]:
await main()